# C1.3 · Cognitive vulnerability and elicitation scaling

**Function C — Red Teaming and Security Research with AI → Agentic Evaluation and Red Teaming**

Builds on **[C1.2 · Weaponizing the ingestion path](https://spbreed.github.io/cyber-commons/lessons/C1.2.html)**.

| | |
|---|---|
| Tools used | Inspect |

## What this lesson is

**What it covers.** Elicitation techniques that scale — cross-prompt attention degradation, context crowding — that strip safety while retaining tool use, measured on reproduction rather than on a transcript.

**Why a security engineer needs it.** A jailbreak that reproduces once is an anecdote; one with a measured success rate across attempts and seeds is a finding. The retained-tool-use combination is the one that turns a bypass into an incident.

## 1 · The hook

A jailbreak that strips safety and leaves tool use intact is the dangerous combination, because a model that cannot act is a curiosity and one that can is an incident. And it only counts if it reproduces across attempts, not in one screenshot.

> **At CyberTravels.** The target is CyberTravels' advisor, and the dangerous outcome is a jailbreak that keeps the booking and refund tools while shedding the policy that governed them.

## 2 · The framework

```
   one transcript              a technique

   works once  -> anecdote     run N times, vary seed and wording
                                        |
                                        v
                               reproduction rate = 0.0 .. 1.0
                               with a denominator

   the dangerous shape: safety stripped, tool use RETAINED.
   a jailbreak that cannot act is a curiosity; one that can is an incident.
```

Elicitation is the model-layer attack: not a single clever prompt, but a
**technique that scales**. Cross-prompt attention degradation, context-window
crowding and instruction-hierarchy confusion all strip a model's safety
behaviour while leaving its tool-use capability intact — which is the dangerous
combination, because a jailbroken model that cannot act is a curiosity and one
that can is an incident.

Research here is judged on reproducibility, not on a single transcript. A
jailbreak that works once is an anecdote; one that reproduces across attempts,
seeds and minor rewordings is a finding with a measurable success rate.

## 3 · The procedure, as a skill

The skill runs a candidate elicitation technique against CyberTravels' advisor repeatedly and reports the share of attempts that reproduced — the difference between a finding and a lucky transcript.

### The skill — [`skills/research/technique-reproducibility-test/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/technique-reproducibility-test/SKILL.md)

```yaml
name: technique-reproducibility-test
description: >-
  Run a technique enough times to say whether it reproduces, and compute the
  sample size a before-and-after comparison actually needed before claiming a
  control worked. Use when a jailbreak "works", or when a fix is declared
  effective from a handful of trials.
allowed-tools: Read, Grep, Glob
```

# One success is an anecdote; the interval is the result

Model-layer research is statistical whether or not anybody does the statistics.
A technique that succeeds once may not reproduce; a control that appears to
help at n=20 has an interval overlapping the baseline. Both errors are avoided
by the same discipline — report a rate with an interval, and compute the sample
size before running the comparison.

## When to use this

Any claim about model behaviour: a technique that works, a control that helps, a
model that is safer than another.

## Procedure

**1 — Define success mechanically.** A string, a state, a check — something a
script decides. "The model complied" judged by reading is not reproducible
between two people.

**2 — Run enough trials to produce a rate, and hold the conditions fixed.**
Same model version, same temperature, same prompt. Record the version: a rate
without one is unrepeatable by construction.

**3 — Classify the technique honestly.** Not reproduced, flaky, or reproducible.
Flaky is a real and common answer and it deserves the word rather than a
rounded-up rate.

**4 — Compute the required sample size before comparing.** From the baseline
rate, the effect you would care about, and the power you want. Then run that
many. Doing this afterwards produces the number that makes the result you got
look significant.

**5 — Report intervals, and say when they overlap.** Show the same true effect
at n=20, n=100 and n=1000 if you need to make the point: the effect did not
change, the ability to see it did.

## Example

**Input** — the fixture committed at the top of [`scripts/technique_reproducibility_test.py`](scripts/technique_reproducibility_test.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
technique              rate              ci95  verdict
------------------------------------------------------------
direct override       0.055    (0.023, 0.087)  flaky
context reframe       0.395    (0.327, 0.463)  flaky
task nesting          0.660    (0.594, 0.726)  reproducible

'It worked' is true for all three. Only one is reproducible.
     n            before             after  conclusion
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "success_criterion": "str",
  "conditions": {"model": "str", "version": "str", "temperature": 0.0},
  "techniques": [{"name": "str", "trials": 0, "rate": 0.0, "interval": [0.0, 0.0],
                  "verdict": "not reproduced|flaky|reproducible"}],
  "comparison": {"before": 0.0, "after": 0.0, "n": 0, "required_n": 0, "separated": false}
}
```

## Failure modes

- **Reporting a rate with no model version.** Nobody can repeat it.
- **Computing the sample size afterwards.** That is choosing the number that
  fits.
- **Rounding flaky up to works.** It is the finding, not a rough edge.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/technique-reproducibility-test/scripts/technique_reproducibility_test.py
SCRIPT = "skills/research/technique-reproducibility-test/scripts/technique_reproducibility_test.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The technique reproduces on a measurable fraction of attempts rather than all or none, and the report is that rate with its denominator, not the single best transcript.

## Your turn

Take any jailbreak you have seen shared as a screenshot and run it twenty times. The reproduction rate is the finding; the screenshot was marketing.

---

**Next → [C1.4 · Establishing telemetry and detecting the actor](https://spbreed.github.io/cyber-commons/lessons/C1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*